# Volta River Basin — Crop Type Mapping & Agricultural Intelligence
## PyGeoVision · Sentinel-2 Multi-Temporal · Ghana
---
**Study area:** Lower Volta River Basin agricultural zone, Eastern Ghana  
**Problem:** Ghana's food security depends on smallholder rice and maize cultivation
along the Volta River floodplain. Irregular rainfall, flooding, and land-use change
threaten crop yields. Early-season crop type identification enables targeted
agricultural extension support.  
**Approach:** Multi-temporal Sentinel-2 NDVI/EVI phenology → crop type segmentation →
yield-risk zone mapping  
**Real AOI:** Lower Volta floodplain polygon (Volta River estuary, Eastern Region)


In [ ]:
# ── Cell 1: Imports & Setup ──────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import pathlib, json, time, datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

try:
    import torch; import torch.nn as nn
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    HAS_TORCH = True
except ImportError:
    torch = nn = None; DEVICE = "cpu"; HAS_TORCH = False

import rasterio
import pygeovision as pgv
from pygeovision import PyGeoVision

client = PyGeoVision()
print(f"pygeovision  : {pgv.__version__}")
print(f"Device       : {DEVICE}")


In [ ]:
# ── Cell 2: Study Area — Lower Volta Floodplain AOI ──────────────────────────
# Lower Volta River agricultural polygon
# Source: Volta Basin Authority data + Ghana EPA Environmental Impact zones
# Area: ~440 km²  |  CRS: WGS84

VOLTA_AOI = {
    "type": "Polygon",
    "coordinates": [[
        [-0.2000, 6.0500],
        [ 0.0500, 6.0500],
        [ 0.0500, 6.2500],
        [-0.2000, 6.2500],
        [-0.2000, 6.0500],
    ]]
}

coords     = VOLTA_AOI["coordinates"][0]
VOLTA_BBOX = (min(c[0] for c in coords), min(c[1] for c in coords),
              max(c[0] for c in coords), max(c[1] for c in coords))

# Phenology windows — 3 critical crop growth stages for tropical Ghana
# Wet season (major): planting Mar-Apr, vegetative May-Jun, harvest Jul-Aug
# Dry season (minor): planting Oct-Nov, harvest Jan-Feb
PHENO_WINDOWS = {
    "planting"   : ("2024-03-01", "2024-04-30"),   # bare soil / early growth
    "vegetative" : ("2024-05-01", "2024-06-30"),   # peak greenness
    "harvest"    : ("2024-07-01", "2024-08-31"),   # senescence / harvest
}

BANDS_S2   = ["B02","B03","B04","B08","B8A","B11","B12"]
OUTPUT_DIR = pathlib.Path("./volta_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AOI_PATH = OUTPUT_DIR / "volta_aoi.geojson"
with open(AOI_PATH, "w") as f:
    json.dump({"type":"FeatureCollection","features":[{"type":"Feature","geometry":VOLTA_AOI,"properties":{}}]}, f)

print("Lower Volta Agricultural Zone")
print(f"  Bbox              : {VOLTA_BBOX}")
print(f"  Area (approx)     : ~440 km²")
print(f"  Phenology windows : {len(PHENO_WINDOWS)}")
for name, (s, e) in PHENO_WINDOWS.items():
    print(f"    {name:12s}: {s} → {e}")


In [ ]:
# ── Cell 3: Search Multi-Temporal Sentinel-2 ─────────────────────────────────
print("Searching Sentinel-2 across all 3 phenology windows…")
print()

SCENES = {}
for stage, (start, end) in PHENO_WINDOWS.items():
    scenes = client.search(
        bbox            = VOLTA_BBOX,
        date_range      = (start, end),
        satellite       = "Sentinel-2",
        providers       = ["planetary_computer"],
        cloud_cover_max = 20,
        max_results     = 8,
        use_cache       = False,
    )
    scenes.sort(key=lambda s: s.cloud_cover or 99)
    SCENES[stage] = scenes[0] if scenes else None
    status = f"{scenes[0].date} cloud={scenes[0].cloud_cover:.0f}%" if scenes else "NOT FOUND"
    print(f"  {stage:12s}: {len(scenes)} found → selected {status}")

print()
print("Multi-temporal stack ready for crop phenology analysis.")


In [ ]:
# ── Cell 4: Download All Phenology Scenes ────────────────────────────────────
print("Downloading 3 phenology scenes…")
RAW_DIR = OUTPUT_DIR / "raw"

DOWNLOADS = {}
for stage, scene in SCENES.items():
    if scene is None:
        print(f"  SKIP {stage}: no scene found"); continue
    dls = client.download(
        [scene],
        output_dir   = str(RAW_DIR / stage),
        bands        = BANDS_S2,
        post_process = ["reproject:EPSG:32630", "cog"],
    )
    ok = next((d for d in dls if d.success and d.path), None)
    DOWNLOADS[stage] = str(ok.path) if ok else None
    if ok:
        sz = pathlib.Path(ok.path).stat().st_size / 1e6
        print(f"  ✓ {stage:12s}: {pathlib.Path(ok.path).name}  ({sz:.1f} MB)")
    else:
        print(f"  ✗ {stage:12s}: download failed")


In [ ]:
# ── Cell 5: Preprocess All Scenes → AI-Ready ─────────────────────────────────
print("Preprocessing 3 scenes through DataValidator…")
PREP_DIR = OUTPUT_DIR / "preprocessed"
PREP_DIR.mkdir(exist_ok=True)

READY = {}
for stage, raw_path in DOWNLOADS.items():
    if not raw_path:
        print(f"  SKIP {stage}"); continue
    out = str(PREP_DIR / f"{stage}_ready.tif")
    result = client.prepare_for_ai(
        raw_path,
        bbox         = VOLTA_BBOX,
        bbox_crs     = "EPSG:4326",
        clip_geojson = str(AOI_PATH),
        normalise    = "scale_factor",
        scale_factor = 10000.0,
        model_type   = "segmentation",
        output_path  = out,
    )
    valid = result.get("report", {}).get("valid", False)
    READY[stage] = out if valid and pathlib.Path(out).exists() else None
    status = f"✓ shape={result.get('shape')}" if valid else f"✗ {result.get('report',{}).get('errors','?')}"
    print(f"  {stage:12s}: {status}")


In [ ]:
# ── Cell 6: Spectral Index Time Series ───────────────────────────────────────
# Compute NDVI and EVI for each phenology window
# NDVI = (NIR-Red)/(NIR+Red)           Band indices in 7-band stack:
# EVI  = 2.5*(NIR-Red)/(NIR+6*Red-7.5*Blue+1)
#   0=Blue, 1=Green, 2=Red, 3=NIR, 4=B8A, 5=SWIR1, 6=SWIR2

print("Computing spectral indices across phenology windows…")
INDEX_DIR = OUTPUT_DIR / "indices"
INDEX_DIR.mkdir(exist_ok=True)

INDICES = {"ndvi": {}, "evi": {}}
eps = 1e-6

for stage, path in READY.items():
    if not path or not pathlib.Path(path).exists():
        print(f"  SKIP {stage}"); continue
    with rasterio.open(path) as src:
        data = src.read().astype("float32")
        meta = src.meta.copy()

    blue = data[0]; green = data[1]; red = data[2]; nir = data[3]

    ndvi = (nir - red) / (nir + red + eps)
    evi  = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1 + eps)
    ndvi = np.clip(ndvi, -1.0, 1.0)
    evi  = np.clip(evi,  -1.0, 2.0)

    for idx_name, arr in [("ndvi", ndvi), ("evi", evi)]:
        out = str(INDEX_DIR / f"{stage}_{idx_name}.tif")
        meta.update(count=1, dtype="float32", nodata=-9999.0)
        with rasterio.open(out, "w", **meta) as dst:
            dst.write(arr[np.newaxis])
        INDICES[idx_name][stage] = arr

    print(f"  ✓ {stage:12s}: NDVI={ndvi.mean():.3f}  EVI={evi.mean():.3f}")

print()
print("Phenological signatures:")
for stage in PHENO_WINDOWS:
    if stage in INDICES["ndvi"]:
        n = INDICES["ndvi"][stage].mean()
        e = INDICES["evi"][stage].mean()
        print(f"  {stage:12s}: NDVI={n:.3f}  EVI={e:.3f}")


In [ ]:
# ── Cell 7: Crop Type Classification from Phenology ───────────────────────────
# Crop type classification using phenological NDVI signatures:
#   Rice    : high vegetative NDVI (0.60+), rapid senescence at harvest
#   Maize   : moderate vegetative NDVI (0.45-0.65), steady phenology
#   Fallow  : low NDVI across all stages (<0.25)
#   Riparian forest: consistently high NDVI (0.70+) all stages

print("Classifying crop types from phenological signatures…")

CROP_LABELS  = {0:"Water/Fallow",1:"Rice",2:"Maize/Sorghum",3:"Mixed Veg",4:"Riparian Forest"}
CROP_COLOURS = ["#3498db","#27ae60","#f39c12","#e74c3c","#1a5276"]

def classify_crops_from_phenology(indices_ndvi):
    """Rule-based phenological crop classifier."""
    stages = ["planting","vegetative","harvest"]
    available = [s for s in stages if s in indices_ndvi]
    if not available:
        return None

    # Use vegetative as primary, fallback to available
    ref   = indices_ndvi.get("vegetative", indices_ndvi[available[0]])
    plant = indices_ndvi.get("planting",   ref)
    harv  = indices_ndvi.get("harvest",    ref)

    H, W = ref.shape
    out  = np.zeros((H, W), dtype="uint8")

    # Align all arrays to same shape
    def crop(arr): return arr[:H, :W]
    ref = crop(ref); plant = crop(plant); harv = crop(harv)

    # Classification rules
    out[ref < 0.15]  = 0   # Water/Fallow
    out[(ref >= 0.15) & (ref < 0.50) & (harv < ref - 0.05)] = 1   # Rice (rapid senescence)
    out[(ref >= 0.30) & (ref < 0.65) & (abs(harv - ref) < 0.08)]  = 2   # Maize (steady)
    out[(ref >= 0.40) & (ref < 0.60) & (plant > 0.30)]             = 3   # Mixed veg
    out[ref >= 0.65]  = 4   # Riparian forest (persistently high)
    return out

CROP_MAP = classify_crops_from_phenology(INDICES["ndvi"])

if CROP_MAP is not None:
    print("Crop type distribution:")
    total = CROP_MAP.size
    for cls_id, label in CROP_LABELS.items():
        pct     = (CROP_MAP == cls_id).mean() * 100
        area_km2 = (CROP_MAP == cls_id).sum() * 100 / 1e6  # 10m pixels
        print(f"  {label:18s}: {pct:5.1f}%  ({area_km2:.1f} km²)")


In [ ]:
# ── Cell 8: Yield Risk Assessment ────────────────────────────────────────────
# Combine NDVI change (vegetative → harvest) with crop type
# to identify high-risk fields for extension services

print("Computing yield risk zones…")

RISK_LABELS  = {0:"Low Risk",1:"Moderate Risk",2:"High Risk",3:"Critical"}
RISK_COLOURS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c"]

RISK_MAP = None
if INDICES["ndvi"].get("vegetative") is not None and INDICES["ndvi"].get("harvest") is not None:
    veg  = INDICES["ndvi"]["vegetative"]
    harv = INDICES["ndvi"]["harvest"]
    H = min(veg.shape[0], harv.shape[0])
    W = min(veg.shape[1], harv.shape[1])
    veg = veg[:H,:W]; harv = harv[:H,:W]

    # Senescence index: large drop = stress / early harvest = yield risk
    senescence = veg - harv

    RISK_MAP = np.zeros((H, W), dtype="uint8")
    RISK_MAP[senescence < 0.05]  = 0   # Low risk: normal
    RISK_MAP[(senescence >= 0.05) & (senescence < 0.15)] = 1   # Moderate
    RISK_MAP[(senescence >= 0.15) & (senescence < 0.25)] = 2   # High
    RISK_MAP[senescence >= 0.25]  = 3   # Critical: severe stress

    print("Yield risk zone distribution:")
    for cls_id, label in RISK_LABELS.items():
        pct = (RISK_MAP == cls_id).mean() * 100
        print(f"  {label:15s}: {pct:.1f}%")
    print()
    print(f"  Fields at high or critical risk: {((RISK_MAP>=2).mean()*100):.1f}%")
    print("  → Recommend immediate agricultural extension visits to high-risk zones")


In [ ]:
# ── Cell 9: Visualisation — 5-Panel Agricultural Intelligence ───────────────
fig, axes = plt.subplots(1, 5, figsize=(28, 6))
fig.patch.set_facecolor("#0D1B2A")
for ax in axes: ax.set_facecolor("#1C2C3C")

CROP_CMAP = mcolors.ListedColormap(CROP_COLOURS)
RISK_CMAP = mcolors.ListedColormap(RISK_COLOURS)

# NDVI per phenology stage
stage_keys = ["planting", "vegetative", "harvest"]
for i, stage in enumerate(stage_keys):
    ax = axes[i]
    arr = INDICES["ndvi"].get(stage)
    if arr is not None:
        im = ax.imshow(arr, cmap="RdYlGn", vmin=-0.1, vmax=0.9, aspect="auto")
        plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02).ax.tick_params(colors="white",labelsize=6)
    ax.set_title(f"NDVI — {stage.title()}", color="white", fontsize=9, fontweight="bold")
    ax.axis("off")

# Crop map
ax = axes[3]
if CROP_MAP is not None:
    im = ax.imshow(CROP_MAP, cmap=CROP_CMAP, vmin=0, vmax=4, aspect="auto")
    patches = [mpatches.Patch(color=c, label=l) for c, l in zip(CROP_COLOURS, CROP_LABELS.values())]
    ax.legend(handles=patches, fontsize=6, loc="lower right",
              facecolor="#0D1B2A", labelcolor="white", framealpha=0.85)
ax.set_title("Crop Type Map", color="white", fontsize=9, fontweight="bold")
ax.axis("off")

# Risk map
ax = axes[4]
if RISK_MAP is not None:
    im = ax.imshow(RISK_MAP, cmap=RISK_CMAP, vmin=0, vmax=3, aspect="auto")
    patches = [mpatches.Patch(color=c, label=l) for c, l in zip(RISK_COLOURS, RISK_LABELS.values())]
    ax.legend(handles=patches, fontsize=6, loc="lower right",
              facecolor="#0D1B2A", labelcolor="white", framealpha=0.85)
ax.set_title("Yield Risk Zones", color="white", fontsize=9, fontweight="bold")
ax.axis("off")

fig.suptitle("Volta River Basin — Crop Type Mapping & Agricultural Intelligence
"
             "Sentinel-2 Multi-Temporal · Phenological Analysis · Eastern Ghana",
             color="white", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "volta_agricultural_intelligence.png"),
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {OUTPUT_DIR}/volta_agricultural_intelligence.png")


In [ ]:
# ── Cell 10: Export & Report ──────────────────────────────────────────────────
# Save crop map and risk map as GeoTIFFs + JSON report

ref_path = next((p for p in READY.values() if p and pathlib.Path(p).exists()), None)
if ref_path:
    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(count=1, dtype="uint8", nodata=255)

    if CROP_MAP is not None:
        with rasterio.open(str(OUTPUT_DIR/"volta_crop_map.tif"), "w", **meta) as dst:
            H,W = CROP_MAP.shape
            dst.write(CROP_MAP[:meta["height"],:meta["width"]][np.newaxis])
        print(f"✓ Crop map saved: {OUTPUT_DIR}/volta_crop_map.tif")

    if RISK_MAP is not None:
        with rasterio.open(str(OUTPUT_DIR/"volta_risk_map.tif"), "w", **meta) as dst:
            H,W = RISK_MAP.shape
            dst.write(RISK_MAP[:meta["height"],:meta["width"]][np.newaxis])
        print(f"✓ Risk map saved: {OUTPUT_DIR}/volta_risk_map.tif")

report = {
    "project"   : "Volta Basin Crop Type Mapping",
    "satellite" : "Sentinel-2",
    "phenology_windows": PHENO_WINDOWS,
    "generated" : datetime.datetime.utcnow().isoformat(),
    "crop_distribution": {
        CROP_LABELS[i]: round(float((CROP_MAP==i).mean()*100),2) if CROP_MAP is not None else 0
        for i in CROP_LABELS
    },
    "risk_distribution": {
        RISK_LABELS[i]: round(float((RISK_MAP==i).mean()*100),2) if RISK_MAP is not None else 0
        for i in RISK_LABELS
    },
    "high_risk_fields_pct": round(float(((RISK_MAP>=2).mean()*100)),2) if RISK_MAP is not None else 0,
}
with open(OUTPUT_DIR/"volta_report.json","w") as f:
    json.dump(report, f, indent=2)
print()
print("═"*55)
print("VOLTA BASIN — AGRICULTURAL INTELLIGENCE REPORT")
print("═"*55)
for k,v in report["crop_distribution"].items():
    print(f"  {k:20s}: {v:.1f}%")
print()
print(f"  High/Critical Risk: {report['high_risk_fields_pct']:.1f}% of agricultural area")
print(f"
Report: {OUTPUT_DIR}/volta_report.json")
